# EcoPack Predictor Accuracy

This notebook evaluates the first prediction-accuracy metric for EcoPack's normalized-runtime predictor.

For each application and GPU count, it reproduces the predictor used in `ecoPack.py`:

Then it compares:
- predicted normalized runtime: `max(predictor_value) / predictor_value`
- true normalized runtime: `runtime / min(runtime)`

Main metrics in this first version:
- MAE
- RMSE


In [29]:
from pathlib import Path
import math
import re

import pandas as pd

DATA_ROOT = Path('/home/ac.zzheng/power/GPGPU/coSched/data')
SYSTEMS = ['V100', 'A100', 'H100']
SECTION_RE = re.compile(r'^===== .*?/([^/ ]+) =====$')
SM_OVERRIDE_APPS = {
    'simpleCUBLASXT',
    'simpleCUFFT_2d_MGPU',
    'simpleCUFFT_MGPU',
    'miniweather',
}
DRAM_SM_THRESHOLD = 0.02


In [30]:
def parse_perf_metrics(metrics_path: Path):
    rows = []
    current_app = None

    for raw_line in metrics_path.read_text().splitlines():
        line = raw_line.strip()
        if not line:
            continue

        match = SECTION_RE.match(line)
        if match:
            current_app = match.group(1)
            continue

        if current_app is None or line.startswith('cap=') or line.startswith('gpu_count'):
            continue

        parts = line.split()
        if len(parts) < 6:
            raise ValueError(f'Expected 6 columns in {metrics_path}, got {len(parts)} for line: {line}')

        rows.append({
            'app': current_app,
            'gpu_count': int(parts[0]),
            'runtime_s': float(parts[1]),
            'avg_power_w': float(parts[2]),
            'dram_sum': float(parts[3]),
            'sm_sum': float(parts[4]),
            'fp_sum': float(parts[5]),
        })

    return pd.DataFrame(rows).sort_values(['app', 'gpu_count']).reset_index(drop=True)


def predictor_name_and_values(app_df: pd.DataFrame):
    app = app_df['app'].iloc[0]

    sm_per_gpu = {
        int(row.gpu_count): row.sm_sum / float(row.gpu_count)
        for row in app_df.itertuples(index=False)
    }
    if app in SM_OVERRIDE_APPS:
        return 'sm_sum/gpu_count (override)', sm_per_gpu

    dram_values = dict(zip(app_df['gpu_count'], app_df['dram_sum']))
    if any(value < DRAM_SM_THRESHOLD for value in dram_values.values()):
        return f'sm_sum/gpu_count (dram<{DRAM_SM_THRESHOLD})', sm_per_gpu
    if any(value > 0.0 for value in dram_values.values()):
        return 'dram_sum', dram_values

    fp_per_gpu = {
        int(row.gpu_count): row.fp_sum / float(row.gpu_count)
        for row in app_df.itertuples(index=False)
    }
    if any(value > 0.0 for value in fp_per_gpu.values()):
        return 'fp_sum/gpu_count', fp_per_gpu

    return 'sm_sum/gpu_count', sm_per_gpu


def predicted_norm_runtime(predictor_values):
    max_value = max(predictor_values.values())
    min_value = min(predictor_values.values())

    if max_value <= 0.0:
        return {gpu_count: 1.0 for gpu_count in predictor_values}

    if abs(max_value - min_value) < 1e-12:
        return {gpu_count: 1.0 for gpu_count in predictor_values}

    result = {}
    for gpu_count, value in predictor_values.items():
        if value <= 0.0:
            raise ValueError(f'Predictor value must be positive, got {value} for {gpu_count} GPUs')
        result[gpu_count] = max_value / value
    return result


def build_prediction_frame(system: str):
    metrics_path = DATA_ROOT / system / 'perf_metrics.txt'
    df = parse_perf_metrics(metrics_path)
    rows = []

    for app, app_df in df.groupby('app', sort=True):
        predictor_name, predictor_values = predictor_name_and_values(app_df)
        pred_nrt = predicted_norm_runtime(predictor_values)
        min_runtime = app_df['runtime_s'].min()

        for row in app_df.itertuples(index=False):
            true_nrt = row.runtime_s / min_runtime
            pred = pred_nrt[int(row.gpu_count)]
            rows.append({
                'system': system,
                'app': app,
                'gpu_count': int(row.gpu_count),
                'runtime_s': row.runtime_s,
                'true_norm_runtime': true_nrt,
                'pred_norm_runtime': pred,
                'abs_error': abs(pred - true_nrt),
                'sq_error': (pred - true_nrt) ** 2,
                'predictor': predictor_name,
            })

    return pd.DataFrame(rows).sort_values(['app', 'gpu_count']).reset_index(drop=True)


In [31]:
prediction_frames = {system: build_prediction_frame(system) for system in SYSTEMS}
all_predictions = pd.concat(prediction_frames.values(), ignore_index=True)

summary_rows = []
for system, frame in prediction_frames.items():
    summary_rows.append({
        'system': system,
        'num_apps': frame['app'].nunique(),
        'num_points': len(frame),
        'mae': frame['abs_error'].mean(),
        'rmse': math.sqrt(frame['sq_error'].mean()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('system').reset_index(drop=True)
summary_df


,system,num_apps,num_points,mae,rmse
0,A100,22,81,0.481823,1.500780
1,H100,22,80,0.284649,0.586114
2,V100,22,80,0.421842,0.926219


In [32]:
app_summary_rows = []
for system, frame in prediction_frames.items():
    grouped = frame.groupby('app', sort=True)
    for app, app_df in grouped:
        app_summary_rows.append({
            'system': system,
            'app': app,
            'num_points': len(app_df),
            'predictor': app_df['predictor'].iloc[0],
            'mae': app_df['abs_error'].mean(),
            'rmse': math.sqrt(app_df['sq_error'].mean()),
        })

app_summary_df = pd.DataFrame(app_summary_rows).sort_values(['system', 'rmse', 'app']).reset_index(drop=True)
# app_summary_df


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

mae_df = app_summary_df[['system', 'app', 'mae']].copy()
system_order = ['V100', 'A100', 'H100']
colors = ['#4C78A8', '#F58518', '#54A24B']

data = [mae_df.loc[mae_df['system'] == s, 'mae'].values for s in system_order]

fig, ax = plt.subplots(figsize=(6, 4.5))

bp = ax.boxplot(
    data, patch_artist=True, showmeans=True, widths=0.45,
    flierprops=dict(marker='', linewidth=0),  # hide default outlier markers
    meanprops=dict(marker='D', markerfacecolor='white', markeredgecolor='black', markersize=5, zorder=4),
    medianprops=dict(color='white', linewidth=2),
    whiskerprops=dict(color='#333333', linewidth=1.2),
    capprops=dict(color='#333333', linewidth=1.2),
    boxprops=dict(linewidth=1.2),
)

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('#333333')
    patch.set_alpha(0.75)

# Overlay individual data points with jitter
rng = np.random.default_rng(42)
for idx, system in enumerate(system_order):
    subset = mae_df[mae_df['system'] == system]
    jitter = rng.uniform(-0.12, 0.12, size=len(subset))
    ax.scatter(idx + 1 + jitter, subset['mae'], color='#222222', s=18,
               alpha=0.4, zorder=3, edgecolors='none')

    # Label outliers
    q1 = subset['mae'].quantile(0.25)
    q3 = subset['mae'].quantile(0.75)
    cutoff = q3 + 1.5 * (q3 - q1)
    outliers = subset[subset['mae'] > cutoff]
    for _, row in outliers.iterrows():
        ax.annotate(
            row['app'], (idx + 1, row['mae']),
            xytext=(8, 0), textcoords='offset points',
            fontsize=7.5, va='center', fontstyle='italic', color='#444444',
        )

ax.set_xticks(range(1, len(system_order) + 1))
ax.set_xticklabels(system_order, fontweight='bold')
ax.set_ylabel('Per-app MAE', fontsize=12)
ax.set_title('EcoPack Predictor Accuracy by System', fontsize=13, fontweight='bold', pad=10)
ax.grid(axis='y', linestyle='--', alpha=0.25, color='#999999')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

mae_df.sort_values(['system', 'mae'], ascending=[True, False]).reset_index(drop=True)